# Khi mean lừa: nhiều đỉnh, số 0, đuôi dài và vài giá trị cực trị

Module 08 đã cho thấy mean bị đuôi dài kéo lệch trên giá trị hoá đơn. Bài này đi tiếp ba kiểu mean "lừa" khác
mà DA gặp hằng tuần, trên hai bộ dữ liệu:

- **Telco Customer Churn** (`telco-churn`, 7 043 khách): cước tháng `MonthlyCharges` có **nhiều đỉnh**.
- **Rossmann Store Sales** (`store-sales`, 1 115 cửa hàng): doanh số ngày có **số 0 của ngày đóng cửa**;
  khoảng cách tới đối thủ `CompetitionDistance` có **đuôi dài**; doanh số trung bình mỗi cửa hàng có vài
  cửa hàng **cực lớn**.

Việc của bạn: viết bốn công cụ đo "mean có đáng tin không" (trimmed mean, winsorized mean, tỉ lệ quan sát gần
một tâm, mean khi bỏ k giá trị lớn nhất), dùng chúng trên dữ liệu thật để quyết định nên báo con số nào, rồi
**làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, vẽ và in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/basic/03-when-mean-misleads -q`

In [ ]:
# explore: tìm gốc repo, tải dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

telco = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
STORE = ensure("store-sales")
train = pd.read_csv(STORE / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
stores = pd.read_csv(STORE / "store.csv")
print(telco.shape, train.shape, stores.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Trimmed mean và winsorized mean

Hai cách làm mean bớt nhạy với giá trị cực trị:

- `trimmed_mean(s, prop) -> float`: bỏ NaN, sắp tăng dần, bỏ `k = floor(prop * n)` giá trị ở **mỗi** đầu rồi lấy
  mean phần còn lại (khớp `scipy.stats.trim_mean`). `ValueError` nếu `prop` ngoài `[0, 0.5)`.
- `winsorized_mean(s, lower=0.05, upper=0.95) -> float`: bỏ NaN, **thay** giá trị dưới phân vị `lower` bằng
  phân vị đó và trên phân vị `upper` bằng phân vị đó (phân vị theo mặc định của pandas), rồi lấy mean.
  `ValueError` nếu không có `0 <= lower < upper <= 1`.

Trimmed mean vứt bỏ đuôi; winsorized mean giữ số quan sát nhưng "kéo" đuôi về ngưỡng.

In [ ]:
def trimmed_mean(s: pd.Series, prop: float) -> float:
    """Mean sau khi bỏ floor(prop * n) giá trị nhỏ nhất và lớn nhất (bỏ NaN trước)."""
    if not 0 <= prop < 0.5:
        raise ValueError("prop phải nằm trong [0, 0.5)")
    x = np.sort(pd.Series(s).dropna().to_numpy(dtype=float))
    k = int(prop * len(x))
    return float(x[k:len(x) - k].mean())


def winsorized_mean(s: pd.Series, lower: float = 0.05, upper: float = 0.95) -> float:
    """Mean sau khi kẹp giá trị vào [quantile(lower), quantile(upper)] (bỏ NaN trước)."""
    if not 0 <= lower < upper <= 1:
        raise ValueError("cần 0 <= lower < upper <= 1")
    x = pd.Series(s).dropna().astype(float)
    return float(x.clip(x.quantile(lower), x.quantile(upper)).mean())

## 2. Mean có đại diện cho ai không?

`share_within(s, center, width) -> float`: tỉ lệ quan sát (trên phần không NaN) nằm trong đoạn
`[center - width, center + width]`, tính cả hai đầu. Nếu rất ít khách có cước gần mean, "khách trung bình" là
một người không tồn tại.

`mean_without_top(s, k) -> float`: mean sau khi bỏ `k` giá trị lớn nhất (bỏ NaN trước). `ValueError` nếu `k < 0`
hoặc `k >= n`. Nếu bỏ 1% quan sát lớn nhất làm mean đổi nhiều, mean phụ thuộc vào vài điểm, và mọi so sánh dựa
trên mean (tháng này với tháng trước, nhóm A với nhóm B) cũng vậy.

In [ ]:
def share_within(s: pd.Series, center: float, width: float) -> float:
    """Tỉ lệ giá trị không NaN nằm trong [center - width, center + width]."""
    x = pd.Series(s).dropna()
    return float(x.between(center - width, center + width).mean())


def mean_without_top(s: pd.Series, k: int) -> float:
    """Mean sau khi bỏ k giá trị lớn nhất (bỏ NaN trước); ValueError nếu k < 0 hoặc k >= n."""
    x = np.sort(pd.Series(s).dropna().to_numpy(dtype=float))
    if not 0 <= k < len(x):
        raise ValueError("cần 0 <= k < số quan sát")
    return float(x[:len(x) - k].mean())

## 3. Kiểu lừa thứ nhất: nhiều đỉnh

Cước tháng của Telco: mean, median, mode, và tỉ lệ khách có cước gần mean. Rồi tách theo loại internet.

In [ ]:
# explore: cước tháng có mấy đỉnh?
mc = telco.MonthlyCharges
print(f"mean {mc.mean():.2f}, median {mc.median():.2f}, mode {mc.mode().iat[0]:.2f}")
print(f"khách trong mean ± 5 USD: {share_within(mc, mc.mean(), 5):.1%}; trong median ± 5 USD: {share_within(mc, mc.median(), 5):.1%}")
print(telco.groupby("InternetService").MonthlyCharges.agg(["size", "mean", "median", "std"]).round(2))

fig, ax = plt.subplots(figsize=(10, 3.5))
for name, part in telco.groupby("InternetService").MonthlyCharges:
    ax.hist(part, bins=np.arange(15, 125, 2.5), alpha=0.6, label=name)
ax.axvline(mc.mean(), color="black", ls="--", lw=1)
ax.annotate(f"mean {mc.mean():.2f}", (mc.mean(), 0.95), xycoords=("data", "axes fraction"), ha="left", fontsize=9)
ax.legend(title="InternetService", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("MonthlyCharges (USD)")
ax.set_title("Cước tháng: ba nhóm khách, mean nằm giữa hai nhóm", loc="left")
plt.show()

## 4. Kiểu lừa thứ hai: số 0 và tập dòng được tính

`train.csv` có một dòng cho mỗi cửa hàng mỗi ngày, kể cả ngày đóng cửa (`Open = 0`, `Sales = 0`). "Doanh số
trung bình một ngày" phụ thuộc vào việc tính những dòng đó hay không.

In [ ]:
# explore: doanh số ngày, có và không có ngày đóng cửa
open_days = train[(train.Open == 1) & (train.Sales > 0)]
for name, s in [("mọi dòng", train.Sales), ("ngày mở cửa có bán", open_days.Sales)]:
    print(f"{name:>20}: n {len(s):>9,}, mean {s.mean():8,.1f}, median {s.median():8,.1f}, skew {s.skew():.2f}, tỉ lệ 0 {(s == 0).mean():.1%}")

## 5. Kiểu lừa thứ ba: đuôi dài và vài giá trị cực trị

`CompetitionDistance` (mét tới cửa hàng đối thủ gần nhất, 1 112 cửa hàng có số liệu) và doanh số trung bình mỗi
cửa hàng (trên ngày mở cửa). So mean với median, trimmed mean 10%, winsorized mean 5–95%, và mean khi bỏ 10 giá
trị lớn nhất.

In [ ]:
# explore: một bảng "mean có đáng tin không" cho hai biến
store_mean = open_days.groupby("Store").Sales.mean()
rows = {}
for name, s in [("CompetitionDistance", stores.CompetitionDistance), ("doanh số TB/cửa hàng", store_mean)]:
    s = s.dropna()
    rows[name] = {
        "n": len(s), "mean": s.mean(), "median": s.median(), "trimmed_10%": trimmed_mean(s, 0.1),
        "winsorized_5_95": winsorized_mean(s), "mean_bỏ_top_10": mean_without_top(s, 10),
        "skew": s.skew(), "tỉ_lệ_>_mean": (s > s.mean()).mean(),
    }
pd.DataFrame(rows).round(3)

## 6. Quyết định và trắc nghiệm

Với mỗi trường hợp, bạn sẽ báo stakeholder con số nào?

1. Marketing hỏi "khách trả bao nhiêu tiền cước mỗi tháng?" để thiết kế gói mới.
2. Head of Ops hỏi "một ngày bán hàng bình thường của một cửa hàng?" và Finance hỏi "dự báo doanh số tháng tới
   của cả chuỗi?".
3. Team mở rộng hỏi "cửa hàng của mình thường cách đối thủ bao xa?".

Xong thì mở app, làm phần **trắc nghiệm** của bài.